In [1]:
from transformers import AutoModelForSequenceClassification
#from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from scipy.special import softmax
import csv
import urllib.request
import re
from datasets import load_dataset
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import plotly.express as px

# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
 
 
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

# Tasks:
# emoji, emotion, hate, irony, offensive, sentiment
# stance/abortion, stance/atheism, stance/climate, stance/feminist, stance/hillary

task='irony'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# download label mapping
labels=[]
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode('utf-8').split("\n")
    csvreader = csv.reader(html, delimiter='\t')
labels = [row[1] for row in csvreader if len(row) > 1]

# PT
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.save_pretrained(MODEL)

text = "Omg you are so good in doing things bad."
text = preprocess(text)
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)
scores = output[0][0].detach().numpy()
scores = softmax(scores)

# TF
# model = TFAutoModelForSequenceClassification.from_pretrained(MODEL)
# model.save_pretrained(MODEL)

# text = "Good night 😊"
# encoded_input = tokenizer(text, return_tensors='tf')
# output = model(encoded_input)
# scores = output[0][0].numpy()
# scores = softmax(scores)

ranking = np.argsort(scores)
ranking = ranking[::-1]
for i in range(scores.shape[0]):
    l = labels[ranking[i]]
    s = scores[ranking[i]]
    print(f"{i+1}) {l} {np.round(float(s), 4)}")


/home/lukassteu/Master_AI/3rd_Semester/UE_AI_and_Visualization/UE/Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.98s/it]

1) non_irony 0.5627
2) irony 0.4373


In [2]:
tweets = load_dataset("cardiffnlp/tweet_eval", "irony", split="test").to_pandas()
tweets["reference_label"] = tweets["label"].map({
    0: "Non-ironic",
    1: "Ironic",
})

explicit_tag = re.compile(r"(?<!\w)#(?:irony|sarcasm|not)\b", re.IGNORECASE)

tweets["removed_tags"] = tweets["text"].map(
    lambda text: ", ".join(explicit_tag.findall(text))
)
tweets["edited_text"] = tweets["text"].map(
    lambda text: explicit_tag.sub("", text).strip()
)

tweets.groupby(["reference_label", tweets["removed_tags"].ne("")]).size()

reference_label  removed_tags
Ironic           True            311
Non-ironic       False           360
                 True            113
dtype: int64

In [3]:
irony_model_id = "cardiffnlp/twitter-roberta-base-irony"
irony_tokenizer = AutoTokenizer.from_pretrained(irony_model_id)
irony_model = AutoModelForSequenceClassification.from_pretrained(irony_model_id)
irony_model.eval()

def normalize_tweet(text):
    tokens = []
    for token in text.split():
        if token.startswith("@") and len(token) > 1:
            token = "@user"
        elif token.startswith("http"):
            token = "http"
        tokens.append(token)
    return " ".join(tokens)

@torch.inference_mode()
def irony_scores(texts, batch_size=16):
    scores = []
    for start in range(0, len(texts), batch_size):
        batch = [normalize_tweet(text) for text in texts[start:start + batch_size]]
        inputs = irony_tokenizer(
            batch, padding=True, truncation=True, max_length=512,
            return_tensors="pt",
        )
        logits = irony_model(**inputs).logits
        scores.extend(logits.softmax(dim=-1)[:, 1].cpu().tolist())
    return scores

tweets["score_before"] = irony_scores(tweets["text"].tolist())

changed = tweets.loc[tweets["removed_tags"].ne("")].copy()
changed["score_after"] = irony_scores(changed["edited_text"].tolist())
changed["score_delta"] = changed["score_after"] - changed["score_before"]
changed["class_flip"] = changed["score_before"].ge(0.5) != changed["score_after"].ge(0.5)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7460.73it/s]


In [4]:
fig = px.scatter(
    changed,
    x="score_before",
    y="score_after",
    color="reference_label",
    symbol="class_flip",
    hover_data={
        "text": True,
        "edited_text": True,
        "removed_tags": True,
        "score_delta": ":.3f",
    },
    labels={
        "score_before": "Ironie-Score vor dem Entfernen",
        "score_after": "Ironie-Score nach dem Entfernen",
        "reference_label": "Referenzlabel",
        "class_flip": "Klassenwechsel",
    },
    title=f"Einfluss expliziter Ironie-Hashtags · {len(changed)} bearbeitete Tweets",
)

fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
              line=dict(color="gray", dash="dash"))
fig.add_vline(x=0.5, line_dash="dot")
fig.add_hline(y=0.5, line_dash="dot")
fig.update_xaxes(range=[-0.02, 1.02])
fig.update_yaxes(range=[-0.02, 1.02], scaleanchor="x", scaleratio=1)
fig.show()

In [5]:
px.histogram(
    tweets,
    x="score_before",
    facet_col="reference_label",
    histnorm="percent",
    nbins=20,
    title="Ironie-Scores im gesamten Testsplit nach Referenzlabel",
).add_vline(x=0.5, line_dash="dot").show()

# Hier sind die Markergruppen eindeutig. Mehrfachtreffer bleiben im Scatterplot.
single_tag = changed.loc[~changed["removed_tags"].str.contains(",")].copy()
single_tag["marker"] = single_tag["removed_tags"].str.lower()
single_tag["n"] = single_tag.groupby(
    ["marker", "reference_label"]
)["score_delta"].transform("size")

px.box(
    single_tag,
    x="marker",
    y="score_delta",
    color="reference_label",
    hover_data=["n"],
    title="Änderung des Ironie-Scores bei Tweets mit genau einem Tag-Treffer",
).add_hline(y=0, line_dash="dot").show()

changed.loc[
    changed["score_before"].ge(0.5) & changed["score_after"].lt(0.5),
    ["text", "edited_text", "removed_tags", "reference_label",
     "score_before", "score_after", "score_delta"],
].sort_values("score_delta").head(10)

,text,edited_text,removed_tags,reference_label,score_before,score_after,score_delta
